# Agentic RAG and tool boundaries

Agentic RAG combines retrieval with tools and state. This notebook routes knowledge questions to retrieval, consequential requests to an approval boundary, and ambiguous requests to human escalation.

## Control flow

```mermaid
flowchart TD
 Q[Question] --> P[Planner]
 P -->|knowledge| R[Retrieve]
 P -->|action| A[Approval boundary]
 A -->|approved| T[Tool + verify]
 A -->|denied| X[Stop safely]
 P -->|ambiguous| H[Human escalation]
```

In [ ]:
from examples.advanced.agentic_rag import Route, authorize_tool, execute, plan

knowledge = plan('How do I rotate an API key?')
assert knowledge.route == Route.RETRIEVE
print(execute(knowledge), knowledge.trace)

action = plan('Refund the last invoice')
print(execute(action), action.trace)
authorize_tool(action, approved=True)
print(execute(action), action.trace)

In [ ]:
denied = plan('Delete my account')
authorize_tool(denied, approved=False)
assert execute(denied) == 'human-approval-required'
assert 'tool-blocked:approval-missing' in denied.trace

## Exercise

Add a read-only tool, a destructive tool, a turn budget, and a receipt-verification step. Explain which controls remain outside the model and how you would audit the trace.